### DATA INGESTION


In [56]:
### Document Structure

from langchain_core.documents import Document

In [57]:
doc=Document(
    page_content="This my first content on document to create rag" ,
    metadata={
        "source":"example.txt",
        " pages":1,
        "author":"Charitha",
        "date_created":"2024-06-10"
          }
)
doc

Document(metadata={'source': 'example.txt', ' pages': 1, 'author': 'Charitha', 'date_created': '2024-06-10'}, page_content='This my first content on document to create rag')

In [58]:
### create a simple txt file
import os
os.makedirs("../data/text_files", exist_ok=True)

In [59]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}
for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [60]:
### TextLoader

from langchain_community.document_loaders import TextLoader 
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [61]:
### DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [62]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\SQL Apna college.pdf', 'file_path': '..\\data\\pdf\\SQL Apna college.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='SQL\n(Notes by Apna College)\nWhat is Database?\nDatabase is a collection of interrelated data.\nWhat is DBMS?\nDBMS (Database Management System) is software used to create, manage, and organize\ndatabases.\nWhat is RDBMS?\n●\nRDBMS (Relational Database Management System) - is a DBMS based on the\nconcept of tables (also called relations).\n●\nData is organized into tables (also known as relations) with rows (records) and\ncolumns (attributes).\n●\nEg - MySQL, PostgreSQL, Oracle etc.\nWhat is SQL?\nSQL is Structured Query Language - used to store, manipulate and retrieve data from\nRDBMS.\n(It is not a database, it is a language us

In [63]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [64]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [65]:
chunks=split_documents(pdf_documents)
chunks

Split 29 documents into 48 chunks

Example chunk:
Content: SQL
(Notes by Apna College)
What is Database?
Database is a collection of interrelated data.
What is DBMS?
DBMS (Database Management System) is software used to create, manage, and organize
databases....
Metadata: {'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\SQL Apna college.pdf', 'file_path': '..\\data\\pdf\\SQL Apna college.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\SQL Apna college.pdf', 'file_path': '..\\data\\pdf\\SQL Apna college.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='SQL\n(Notes by Apna College)\nWhat is Database?\nDatabase is a collection of interrelated data.\nWhat is DBMS?\nDBMS (Database Management System) is software used to create, manage, and organize\ndatabases.\nWhat is RDBMS?\n●\nRDBMS (Relational Database Management System) - is a DBMS based on the\nconcept of tables (also called relations).\n●\nData is organized into tables (also known as relations) with rows (records) and\ncolumns (attributes).\n●\nEg - MySQL, PostgreSQL, Oracle etc.\nWhat is SQL?\nSQL is Structured Query Language - used to store, manipulate and retrieve data from\nRDBMS.\n(It is not a database, it is a language us

Embedding and VectorStore DB


In [66]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [67]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


VectorStoreDB

In [68]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 144


In [69]:
chunks

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\SQL Apna college.pdf', 'file_path': '..\\data\\pdf\\SQL Apna college.pdf', 'total_pages': 29, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='SQL\n(Notes by Apna College)\nWhat is Database?\nDatabase is a collection of interrelated data.\nWhat is DBMS?\nDBMS (Database Management System) is software used to create, manage, and organize\ndatabases.\nWhat is RDBMS?\n●\nRDBMS (Relational Database Management System) - is a DBMS based on the\nconcept of tables (also called relations).\n●\nData is organized into tables (also known as relations) with rows (records) and\ncolumns (attributes).\n●\nEg - MySQL, PostgreSQL, Oracle etc.\nWhat is SQL?\nSQL is Structured Query Language - used to store, manipulate and retrieve data from\nRDBMS.\n(It is not a database, it is a language us

In [70]:
### Text to Embeddings
texts=[chunk.page_content for chunk in chunks]
### generate embeddings for the chunks
embeddings=embedding_manager.generate_embeddings(texts)
### Strore in the vector store
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 48 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generated embeddings with shape: (48, 384)
Adding 48 documents to vector store...
Successfully added 48 documents to vector store
Total documents in collection: 192


In [71]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [72]:
rag_retriever.retrieve("sql")

Retrieving documents for query: 'sql'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_317d3b40_1',
  'content': '●\nREAD - To read data present in the database.\n●\nUPDATE - Modify already inserted data.\n●\nDELETE - Delete database, table or specific data point/tuple/row or multiple rows.\n*Note - SQL keywords are NOT case sensitive. Eg: select is the same as SELECT in SQL.\nSQL v/s MySQL\nSQL is a language used to perform CRUD operations in Relational DB, while MySQL is a\nRDBMS that uses SQL.',
  'metadata': {'creationdate': '',
   'modDate': '',
   'creator': '',
   'keywords': '',
   'format': 'PDF 1.4',
   'content_length': 378,
   'source': '..\\data\\pdf\\SQL Apna college.pdf',
   'trapped': '',
   'producer': '',
   'creationDate': '',
   'subject': '',
   'page': 0,
   'file_path': '..\\data\\pdf\\SQL Apna college.pdf',
   'moddate': '',
   'doc_index': 1,
   'title': '',
   'author': '',
   'total_pages': 29},
  'similarity_score': 0.03200364112854004,
  'distance': 0.96799635887146,
  'rank': 1},
 {'id': 'doc_4a660616_1',
  'content': '●\nREAD -

INTEGRATING VECTORDB CONTEXT PIPELINE WITH LLM OUTPUT

In [73]:
#### Simple RAG Pipeline with Groq LLM
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("groq"))
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
llm=ChatGroq(groq_api_key=os.getenv("groq"),model="llama-3.1-8b-instant",temperature=0.7,max_tokens=500)
def rag_simple(query,retriever=rag_retriever, llm=llm, top_k=3):
    """Simple RAG pipeline that retrieves documents and generates a response using LLM"""
    # Step 1: Retrieve relevant documents
    retrieved_docs=retriever.retrieve(query,top_k=top_k)
    
    if not retrieved_docs:
        return "No relevant documents found for the query."
    
    # Step 2: Prepare context from retrieved documents
    context="\n\n".join([ doc['content'] for doc in retrieved_docs]) if retrieved_docs else "No context available."
    
    # Step 3: Create prompt for LLM
    prompt="""You are a helpful assistant. Use the following context to answer the question.
        Context:
        {context}
        Question: {query}
        Answer:"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


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


In [74]:
from groq import Groq

# Fix the Groq model error by using a model available in your account.
# If the default model is unavailable, retry with a known Groq model.
try:
    answer = rag_simple("What is SQL?", retriever=rag_retriever, llm=llm)
except Exception as e:
    msg = str(e)
    if "does not exist" in msg or "do not have access" in msg or "model_not_found" in msg:
        print("The configured Groq model is unavailable in this account. Retrying with a supported model...")
        llm = ChatGroq(
            groq_api_key=os.getenv("groq"),
            model="llama-3.3-70b-versatile",
            temperature=0.7,
            max_tokens=500,
        )
        available_models = {
            model.id for model in Groq(api_key=os.getenv("groq")).models.list().data
        }
        preferred_models = [
            "llama-3.3-70b-versatile",
            "llama-3.1-8b-instant",
            "openai/gpt-oss-20b",
            "openai/gpt-oss-120b",
        ]
        model_name = next((name for name in preferred_models if name in available_models), None)

        if model_name is None:
            raise RuntimeError("No preferred chat model is available for this Groq account.")

        llm = ChatGroq(
            groq_api_key=os.getenv("groq"),
            model=model_name,
            temperature=0.7,
            max_tokens=500,
        )

        answer = rag_simple("What is SQL?", retriever=rag_retriever, llm=llm)
       
    else:
        raise


print(answer)

Retrieving documents for query: 'What is SQL?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
The configured Groq model is unavailable in this account. Retrying with a supported model...
Retrieving documents for query: 'What is SQL?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
SQL (Structured Query Language) is a specialized language used to create, manage, and manipulate data in relational database management systems (RDBMS). It allows you to define database structures, insert, query, update, and delete data, and perform other administrative tasks—essentially serving as the interface between you and the database.


Enhanced RAG Pipeline Features

In [76]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what is SQL?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is SQL?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: SQL (Structured Query Language) is a language used to create, manipulate, and retrieve data from relational database management systems (RDBMS). It is not a database itself but the interface for interacting with one.
Sources: [{'source': '..\\data\\pdf\\SQL Apna college.pdf', 'page': 0, 'score': 0.4086291790008545, 'preview': 'SQL\n(Notes by Apna College)\nWhat is Database?\nDatabase is a collection of interrelated data.\nWhat is DBMS?\nDBMS (Database Management System) is software used to create, manage, and organize\ndatabases.\nWhat is RDBMS?\n●\nRDBMS (Relational Database Management System) - is a DBMS based on the\nconcept of ...'}, {'source': '..\\data\\pdf\\SQL Apna college.pdf', 'page': 0, 'score': 0.4086291790008545, 'preview': 'SQL\n(Notes by Apna College)\nWhat is Database?\nDatabase is a collection of interrelated data.\nWhat is DBMS?\nDBMS (Database Management System) is software used

In [77]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is SQL?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is SQL?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
SQL
(Notes by Apna College)
What is Database?
Database is a collection of interrelated data.
What is DBMS?
DBMS (Database Management System) is software used to create, manage, and organize
databases.
What is RDBMS?
●
RDBMS (Relational Database Management System) - is a DBMS based on the
concept of tables (also called relations).
●
Data is organized into tables (also known as relations) with rows (records) and
columns (attributes).
●
Eg - MySQL, PostgreSQL, Oracle etc.
What is SQL?
SQL is Structured Query Language - used to store, manipulate and retrieve data from
RDBMS.
(It is not a database, it is a language used to interact with database)
We use SQL for CRUD Operations :
●
CREATE - To create databases, tables, insert tuples in tables etc
●
READ - To read data present in the database.
●
UPDATE - Modify already inserte